# 🚀 DEMO THỰC HÀNH: DELTA LAKE CHANGE DATA FEED (CDF) TRÊN MICROSOFT FABRIC

## 📌 Giới thiệu về Delta Change Data Feed (CDF)
**Delta Lake Change Data Feed (CDF)** cho phép ghi lại chi tiết mọi biến động dữ liệu ở cấp độ dòng (**Row-level Changes**) giữa các phiên bản (versions) của một bảng Delta. Khi CDF được kích hoạt, Delta Lake sẽ tự động ghi nhận các cột metadata ẩn:
- `_change_type`: Loại thao tác (`insert`, `update_preimage`, `update_postimage`, `delete`).
- `_commit_version`: Số phiên bản commit phát sinh thay đổi.
- `_commit_timestamp`: Thời điểm commit dữ liệu.

### 💡 Tại sao CDF vượt trội hơn so với `updated_at` Timestamp?
1. **Bắt được thao tác DELETE:** Lọc theo cột `updated_at` chỉ thấy các bản ghi mới/sửa, hoàn toàn bỏ sót bản ghi vừa bị xóa khỏi nguồn. CDF ghi nhận sự kiện `delete` rõ ràng.
2. **Theo dõi chính xác trạng thái Trước/Sau khi Update (Audit Trail):** CDF cung cấp cả `update_preimage` (giá trị cũ) và `update_postimage` (giá trị mới).
3. **Hỗ trợ Incremental Sync tối ưu:** Các pipeline downstream (Silver -> Gold / Warehouse / ClickHouse) chỉ cần đọc phần delta thay đổi thay vì scan toàn bộ bảng.

## BƯỚC 1: BẬT KÍCH HOẠT CDF VÀ KHỞI TẠO BẢNG DEMO (VERSION 0)
Chúng ta khởi tạo bảng Delta Lake `lh_silver.dbo.cdf_demo_customers` và kích hoạt thuộc tính `delta.enableChangeDataFeed = true`.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType
from datetime import datetime

TABLE_NAME = "lh_silver.dbo.cdf_demo_customers"

# 1. Định nghĩa dữ liệu ban đầu (Version 0)
initial_data = [
    ("CUST_001", "Nguyen Van A", "STANDARD", "Ha Noi", 1000.0, datetime(2026, 1, 1, 8, 0)),
    ("CUST_002", "Tran Thi B", "GOLD", "Da Nang", 2500.0, datetime(2026, 1, 1, 8, 0)),
    ("CUST_003", "Le Van C", "PLATINUM", "TP HCM", 5000.0, datetime(2026, 1, 1, 8, 0))
]

schema = StructType([
    StructField("customer_id", StringType(), False),
    StructField("customer_name", StringType(), True),
    StructField("tier", StringType(), True),
    StructField("city", StringType(), True),
    StructField("credit_limit", DoubleType(), True),
    StructField("created_at", TimestampType(), True)
])

df_init = spark.createDataFrame(initial_data, schema)

# 2. Ghi bảng Delta và BẬT TÍNH NĂNG CDF
df_init.write.format("delta") \
    .mode("overwrite") \
    .option("delta.enableChangeDataFeed", "true") \
    .saveAsTable(TABLE_NAME)

print(f"✅ Bảng {TABLE_NAME} đã được khởi tạo thành công với tính năng CDF!")
spark.table(TABLE_NAME).show()

## BƯỚC 2: THỰC THI CÁC THAO TÁC INSERT, UPDATE, DELETE QUA NHIỀU TRANSACTION
Mỗi câu lệnh dưới đây sẽ tạo ra 1 Version commit mới trong Transaction Log của Delta Lake.

In [ ]:
# Thao tác 1 (Version 1): INSERT thêm khách hàng mới
spark.sql(f"""
INSERT INTO {TABLE_NAME} 
VALUES ('CUST_004', 'Pham Van D', 'SILVER', 'Can Tho', 1500.0, CURRENT_TIMESTAMP())
""")
print("✅ Version 1: Da INSERT CUST_004")

In [ ]:
# Thao tác 2 (Version 2): UPDATE thông tin của CUST_001 (Nâng tier & tăng hạn mức tín dụng)
spark.sql(f"""
UPDATE {TABLE_NAME}
SET tier = 'VIP_GOLD', credit_limit = 3500.0, city = 'Hai Phong'
WHERE customer_id = 'CUST_001'
""")
print("✅ Version 2: Da UPDATE CUST_001")

In [ ]:
# Thao tác 3 (Version 3): DELETE khách hàng CUST_002 (Xóa hẳn khỏi bảng)
spark.sql(f"""
DELETE FROM {TABLE_NAME}
WHERE customer_id = 'CUST_002'
""")
print("✅ Version 3: Da DELETE CUST_002")

## BƯỚC 3: ĐỌC VÀ TRÍCH XUẤT CHANGE DATA FEED (BATCH READ)
Bây giờ chúng ta sử dụng CDF để xem toàn bộ lịch sử biến động từ phiên bản 0 đến nay.

In [ ]:
# CÁCH 1: Đọc bằng PySpark DataFrame API (option readChangeFeed)
cdf_df = spark.read.format("delta") \
    .option("readChangeFeed", "true") \
    .option("startingVersion", 0) \
    .table(TABLE_NAME)

print("--- BẢNG CHANGE DATA FEED TỪ VERSION 0 ---")
cdf_df.select(
    "customer_id", "customer_name", "tier", "credit_limit",
    "_change_type", "_commit_version", "_commit_timestamp"
).sort("_commit_version", "customer_id").show(truncate=False)

In [ ]:
# CÁCH 2: Đọc bằng Spark SQL table_changes()
display(spark.sql(f"""
SELECT customer_id, customer_name, tier, credit_limit, _change_type, _commit_version, _commit_timestamp
FROM table_changes('{TABLE_NAME}', 1, 3)
ORDER BY _commit_version, customer_id
"""))

## BƯỚC 4: ỨNG DỤNG CDF ĐỂ XÂY DỰNG BẢNG AUDIT LOG THAY ĐỔI GIÁ TRỊ (BEFORE/AFTER)
CDF cực kỳ hữu ích khi bạn cần ghi vết lại lịch sử thay đổi của các trường quan trọng (Ví dụ: Tier, Credit Limit).

In [ ]:
# Lọc các sự kiện UPDATE để so sánh Pre-image (Trước) và Post-image (Sau)
updates_cdf = spark.read.format("delta") \
    .option("readChangeFeed", "true") \
    .option("startingVersion", 1) \
    .table(TABLE_NAME) \
    .filter("_change_type IN ('update_preimage', 'update_postimage')")

updates_cdf.select("customer_id", "tier", "credit_limit", "_change_type", "_commit_version").show()

## BƯỚC 5: HIỂU VÀ SỬ DỤNG STRUCTURED STREAMING VỚI CHANGE DATA FEED

### ⚠️ Tại sao gọi `display(stream_cdf)` hay `stream_cdf.show()` bị lỗi?
Khi dùng `spark.readStream`, DataFrame trở thành **Streaming DataFrame (Unbounded Table)** – một luồng dữ liệu liên tục không có điểm dừng. 
Spark **KHÔNG CHO PHÉP** gọi trực tiếp các hành động Batch như `.show()`, `.collect()`, hoặc `display()` mà phải định nghĩa một **Sink** đầu ra thông qua phương thức `.writeStream.start()`.

### 🔑 Cấu trúc 3 thành phần của Structured Streaming:
1. **Stream Reader (Source):** `spark.readStream.format("delta").option("readChangeFeed", "true").table(...)`
2. **Stream Transformation:** Các phép lọc `.filter()`, `.select()`.
3. **Stream Writer (Sink):** `.writeStream.format(...).option("checkpointLocation", ...).start()`

### 🌟 Kỹ thuật 1: Ghi luồng Stream CDF vào Memory Sink để kiểm tra trên Notebook UI
Để hiển thị và kiểm thử dữ liệu Streaming trực tiếp trong Notebook mà không bị lỗi, ta sử dụng Memory Sink (`format("memory")`):

In [ ]:
# 1. Đọc stream từ CDF
stream_cdf = spark.readStream.format("delta") \
    .option("readChangeFeed", "true") \
    .option("startingVersion", 0) \
    .table(TABLE_NAME)

# 2. Khởi tạo Stream Job ghi ra Memory Sink có tên 'cdf_memory_view'
query = stream_cdf.writeStream \
    .format("memory") \
    .queryName("cdf_memory_view") \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .start()

# Chờ Stream xử lý xong batch dữ liệu hiện tại
query.awaitTermination()

# 3. Bây giờ bạn có thể query bảng 'cdf_memory_view' bình thường!
print("--- DỮ LIỆU ĐỌC BẰNG STRUCTURED STREAMING (MEMORY SINK) ---")
spark.table("cdf_memory_view").select("customer_id", "customer_name", "tier", "_change_type", "_commit_version").show()

### 🚀 Kỹ thuật 2: Stream Sync CDF từ Silver sang Bảng Audit Log (Trigger AvailableNow / Once)
Trong Production (ETL/ELT), bạn dùng `foreachBatch()` hoặc ghi trực tiếp ra bảng Delta khác với **Checkpointing** để đảm bảo tính sẵn sàng cao và không bị trùng lặp dữ liệu (Exactly-Once Processing):

In [ ]:
AUDIT_TABLE = "lh_silver.dbo.cdf_audit_log_target"
CHECKPOINT_PATH = "Files/checkpoints/cdf_audit_demo"

# Hàm xử lý từng Micro-batch khi stream nạp về
def process_micro_batch(micro_batch_df, batch_id):
    print(f"⚡ Xử lý Micro-Batch #{batch_id} có {micro_batch_df.count()} bản ghi thay đổi")
    # Ghi append bản ghi thay đổi vào bảng Audit Log Target
    micro_batch_df.write.format("delta").mode("append").saveAsTable(AUDIT_TABLE)

# Khởi chạy Continuous Incremental Pipeline bằng Trigger AvailableNow
stream_job = spark.readStream.format("delta") \
    .option("readChangeFeed", "true") \
    .option("startingVersion", 0) \
    .table(TABLE_NAME) \
    .writeStream \
    .foreachBatch(process_micro_batch) \
    .option("checkpointLocation", CHECKPOINT_PATH) \
    .trigger(availableNow=True) \
    .start()

stream_job.awaitTermination()

print("✅ Đã Stream ghi nhận thành công toàn bộ biến động sang Bảng Audit Target!")
spark.table(AUDIT_TABLE).select("customer_id", "tier", "_change_type", "_commit_version").show()

## BƯỚC 6: ỨNG DỤNG CDF – CÁCH XEM TRẠNG THÁI HIỆN TẠI VÀ LỊCH SỬ TỪNG THỜI ĐIỂM (SCD TYPE 2)

> **Bản chất của bảng CDF:**
> Bảng Change Data Feed (ở Bước 3) thực chất là một **cuốn sổ nhật ký giao dịch (Event Log / Audit Trail)** cho Spark đọc ngầm. Do đó, người dùng nhìn vào sẽ thấy khó theo dõi trạng thái hiện tại hoặc lịch sử từng thời điểm.

Dưới đây là 3 cách ứng dụng chuẩn hóa để giải quyết trọn vẹn nhu cầu của người dùng và báo cáo BI:
1. **Xem trạng thái HIỆN TẠI (Current Snapshot / SCD Type 1):** Đọc trực tiếp bảng gốc.
2. **Xem trạng thái QUÁ KHỨ (Point-in-Time):** Sử dụng Delta Time Travel (`versionAsOf` / `timestampAsOf`).
3. **Biến đổi CDF thành Bảng Lịch Sử SCD Type 2:** Tạo ra dòng thời gian rõ ràng với `valid_from`, `valid_to`, `is_current`.

In [ ]:
# ==============================================================================
# 1. XEM TRẠNG THÁI HIỆN TẠI (CURRENT STATE / SNAPSHOT)
# Chỉ cần đọc trực tiếp bảng gốc KHÔNG dùng readChangeFeed
# ==============================================================================
print("=== 1. BẢNG TRẠNG THÁI HIỆN TẠI (SNAPSHOT HIỆN HÀNH) ===")
print("-> CUST_001 mang giá trị mới nhất (VIP_GOLD - 3500.0)")
print("-> CUST_002 đã bị xóa nên hoàn toàn biến mất khỏi bảng:")
spark.table(TABLE_NAME).show(truncate=False)


In [ ]:
# ==============================================================================
# 2. XEM TRẠNG THÁI QUÁ KHỨ TẠI TỪNG THỜI ĐIỂM BẰNG DELTA TIME TRAVEL
# ==============================================================================
print("=== 2. QUAY NGƯỢC THỜI GIAN VỀ VERSION 1 ===")
print("-> Lúc này CUST_001 vẫn là STANDARD (chưa đổi VIP) và CUST_002 vẫn còn tồn tại (chưa bị xóa):")
spark.read.format("delta") \
    .option("versionAsOf", 1) \
    .table(TABLE_NAME) \
    .show(truncate=False)


In [ ]:
# ==============================================================================
# 3. BIẾN ĐỔI TRỰC TIẾP CDF THÀNH BẢNG LỊCH SỬ SCD TYPE 2 HOÀN HẢO
# ==============================================================================
from pyspark.sql.window import Window

# 1. Đọc toàn bộ luồng thay đổi từ CDF
cdf_events = spark.read.format("delta") \
    .option("readChangeFeed", "true") \
    .option("startingVersion", 0) \
    .table(TABLE_NAME)

# 2. Định nghĩa cửa sổ theo từng khách hàng, sắp xếp theo thời gian commit
win_history = Window.partitionBy("customer_id").orderBy("_commit_timestamp")

# 3. Lọc bỏ update_preimage để chỉ lấy mốc bắt đầu hiệu lực thực tế
#    Dùng lead() lấy mốc tiếp theo làm valid_to
df_history_scd2 = cdf_events.filter("_change_type != 'update_preimage'") \
    .withColumn("valid_from", F.col("_commit_timestamp")) \
    .withColumn("valid_to", F.lead("_commit_timestamp").over(win_history)) \
    .withColumn("is_current", F.when(F.col("_change_type") == "delete", False).otherwise(F.col("valid_to").isNull())) \
    .filter(F.col("_change_type") != "delete") \
    .select("customer_id", "customer_name", "tier", "credit_limit", "valid_from", "valid_to", "is_current")

# 4. Ghi ra bảng lịch sử SCD Type 2 trên lh_silver.dbo
HISTORY_TABLE = "lh_silver.dbo.cdf_demo_customers_history_scd2"
df_history_scd2.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(HISTORY_TABLE)

print(f"=== 3. BẢNG LỊCH SỬ SCD TYPE 2 DỄ HIỂU NHẤT CHO NGƯỜI DÙNG & BI ({HISTORY_TABLE}) ===")
spark.table(HISTORY_TABLE).sort("customer_id", "valid_from").show(truncate=False)
